# LegalQA — pipeline smoke test + 10 câu train đầu

Notebook kiểm tra luồng chung trên Kaggle: code/unit test → BM25 → Dense → RRF → reranker → generation/fallback → schema/audit/metric. Nó chạy bốn mode `extractive`, `knn`, `hybrid`, `hybrid_rag` trên đúng 10 mẫu đầu của `train.json`.

Lệnh `validate` luôn loại chính ID đang đánh giá khỏi KNN (`exclude_id`), nên kết quả không bị self-match từ train. Đây là smoke test phát hiện lỗi, không phải validation đại diện và không tự cho phép chạy full 1.000 câu.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path
from typing import Any

REPO_URL = 'https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git'
REPO_BRANCH = 'main'
EXPECTED_COMMIT = ''  # Có thể điền full SHA để khóa đúng bản code cần test.
FORCE_RERUN = False
RUN_UNIT_TESTS = True
MODES = ['extractive', 'knn', 'hybrid', 'hybrid_rag']
TRAIN_LIMIT = 10
SEED = 2026

EMBEDDING_MODEL_ID = 'AITeamVN/Vietnamese_Embedding_v2'
RERANKER_MODEL_ID = 'AITeamVN/Vietnamese_Reranker'
GENERATOR_MODEL_ID = 'AITeamVN/Vi-Qwen2-1.5B-RAG'
MODEL_MARKER = '.legalqa_model.json'
BM25_TOP_K = 50
DENSE_TOP_K = 50
RRF_TOP_K = 50
RERANKER_CANDIDATE_K = 20
RERANK_TOP_K = 3
RERANKER_MAX_LENGTH = 1024
MAX_INPUT_TOKENS = 7168
MAX_NEW_TOKENS = 512
TOKEN_LIMIT_RETRY_TOKENS = 768
MAX_LONG_ANSWER_WORDS = 640

KAGGLE = Path('/kaggle/working').is_dir()
INPUT_ROOT = Path('/kaggle/input') if KAGGLE else Path('.').resolve()
REPO_DIR = Path('/kaggle/working/uit-dsc-2026-task2-legalqa') if KAGGLE else Path('.').resolve()
WORK_ROOT = Path('/kaggle/working/legalqa-pipeline-smoke-train10') if KAGGLE else Path('artifacts/legalqa-pipeline-smoke-train10').resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)

def run_stream(command: list[str], *, cwd: Path | None = None, log_path: Path | None = None) -> None:
    print('$', ' '.join(map(str, command)), flush=True)
    lines: list[str] = []
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
    assert process.stdout is not None
    for line in iter(process.stdout.readline, ''):
        print(line, end='', flush=True)
        lines.append(line)
    process.stdout.close()
    return_code = process.wait()
    if log_path is not None:
        log_path.write_text(''.join(lines), encoding='utf-8')
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

print('Kaggle:', KAGGLE, '| Work:', WORK_ROOT)
print('Modes:', MODES, '| Train samples:', TRAIN_LIMIT)

In [ ]:
# 1) Checkout code, cài dependency và chạy regression trước khi dùng GPU.
if KAGGLE:
    if (REPO_DIR / '.git').is_dir():
        run_stream(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_DIR)
    elif REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} tồn tại nhưng không phải Git repo; không tự xóa.')
    else:
        run_stream(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)])

COMMIT_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True, encoding='utf-8').strip()
print('Commit:', COMMIT_SHA)
if EXPECTED_COMMIT and COMMIT_SHA != EXPECTED_COMMIT:
    raise RuntimeError(f'Sai commit: expected={EXPECTED_COMMIT}, actual={COMMIT_SHA}')

if KAGGLE:
    run_stream([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-generator.txt')], cwd=REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

if RUN_UNIT_TESTS:
    unit_patterns = [
        'test_baseline.py',
        'test_storage_remaining.py',
        'test_generator.py',
        'test_dense_rag.py',
        'test_long_answer_routing.py',
    ]
    for pattern in unit_patterns:
        run_stream(
            [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', pattern],
            cwd=REPO_DIR,
            log_path=WORK_ROOT / f'{Path(pattern).stem}.log',
        )
    print('UNIT GATE: PASS (5 targeted suites)')
else:
    print('SKIP unit tests vì RUN_UNIT_TESTS=False')

In [ ]:
# 2) Tìm train, index và ba model trong Kaggle Input; không rebuild artifact.
INPUT_FILES = [path for path in INPUT_ROOT.rglob('*') if path.is_file()] if INPUT_ROOT.exists() else []
print(f'Đã quét {len(INPUT_FILES):,} input files')

def artifact_rank(path: Path) -> tuple[int, int, str]:
    has_manifest = any((parent / 'legalqa_artifacts.json').is_file() for parent in path.parents)
    return (0 if has_manifest else 1, len(path.parts), str(path))

def find_named(names: set[str], *, repo_fallback: bool = False) -> Path:
    accepted = {name.casefold() for name in names}
    matches = sorted((path for path in INPUT_FILES if path.name.casefold() in accepted), key=artifact_rank)
    if matches:
        return matches[0]
    if repo_fallback:
        for name in names:
            candidate = REPO_DIR / 'data' / name
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(f'Không tìm thấy artifact: {sorted(names)}')

def complete_model(folder: Path) -> bool:
    return (folder / 'config.json').is_file() and (any(folder.glob('*.safetensors')) or any(folder.glob('pytorch_model*.bin')))

def discover_model(repo_id: str) -> dict[str, Any]:
    candidates: list[dict[str, Any]] = []
    for marker in (path for path in INPUT_FILES if path.name == MODEL_MARKER):
        try:
            payload = read_json(marker)
        except Exception:
            continue
        if payload.get('repo_id') == repo_id and payload.get('revision') and complete_model(marker.parent):
            candidates.append({'repo_id': repo_id, 'revision': str(payload['revision']), 'path': marker.parent})
    cache_key = ('models--' + repo_id.replace('/', '--')).casefold()
    for config_path in (path for path in INPUT_FILES if path.name == 'config.json'):
        folder = config_path.parent
        if cache_key in [part.casefold() for part in folder.parts] and folder.parent.name == 'snapshots' and complete_model(folder):
            candidates.append({'repo_id': repo_id, 'revision': folder.name, 'path': folder})
    if not candidates:
        raise FileNotFoundError(f'Thiếu snapshot model có revision: {repo_id}')
    return sorted(candidates, key=lambda item: artifact_rank(Path(item['path'])))[0]

TRAIN_PATH = find_named({'train.json'}, repo_fallback=True)
BM25_SOURCE_PATH = find_named({'legalqa.sqlite'})
DENSE_META_PATH = find_named({'legalqa_dense.meta.json'})
DENSE_INDEX_PATH = Path(str(DENSE_META_PATH).removesuffix('.meta.json'))
DENSE_VECTOR_PATH = next((path for path in (DENSE_INDEX_PATH.with_suffix('.faiss'), DENSE_INDEX_PATH.with_suffix('.npy')) if path.is_file()), None)
if DENSE_VECTOR_PATH is None:
    raise FileNotFoundError(f'Thiếu dense vector cạnh {DENSE_META_PATH}')

MODEL_INFO = {
    'embedding': discover_model(EMBEDDING_MODEL_ID),
    'reranker': discover_model(RERANKER_MODEL_ID),
    'generator': discover_model(GENERATOR_MODEL_ID),
}
HF_HUB_CACHE = WORK_ROOT / 'hf-cache'
for info in MODEL_INFO.values():
    snapshot_link = HF_HUB_CACHE / ('models--' + info['repo_id'].replace('/', '--')) / 'snapshots' / info['revision']
    snapshot_link.parent.mkdir(parents=True, exist_ok=True)
    if not snapshot_link.exists() and not snapshot_link.is_symlink():
        snapshot_link.symlink_to(Path(info['path']).resolve(), target_is_directory=True)
os.environ['HF_HUB_CACHE'] = str(HF_HUB_CACHE)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

LOCAL_DB_PATH = WORK_ROOT / 'legalqa.sqlite'
if not LOCAL_DB_PATH.is_file() or LOCAL_DB_PATH.stat().st_size != BM25_SOURCE_PATH.stat().st_size:
    copying = LOCAL_DB_PATH.with_suffix('.sqlite.copying')
    shutil.copyfile(BM25_SOURCE_PATH, copying)
    copying.replace(LOCAL_DB_PATH)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Cần bật GPU Accelerator trên Kaggle.')
print('GPU:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('Train:', TRAIN_PATH)
print('BM25:', LOCAL_DB_PATH)
print('Dense:', DENSE_INDEX_PATH, '|', DENSE_VECTOR_PATH)
print('Models:', json.dumps({key: {**value, 'path': str(value['path'])} for key, value in MODEL_INFO.items()}, ensure_ascii=False, indent=2))

In [ ]:
# 3) Lấy đúng 10 mẫu đầu theo thứ tự trong train.json.
TRAIN_DATA = read_json(TRAIN_PATH)
if not isinstance(TRAIN_DATA, dict):
    raise ValueError('train.json phải là object keyed by ID')
TRAIN10_ITEMS = list(TRAIN_DATA.items())[:TRAIN_LIMIT]
if len(TRAIN10_ITEMS) != TRAIN_LIMIT:
    raise RuntimeError(f'Train chỉ có {len(TRAIN10_ITEMS)} mẫu, cần {TRAIN_LIMIT}')
for sample_id, item in TRAIN10_ITEMS:
    if not isinstance(item, dict) or not str(item.get('question') or '').strip() or not str(item.get('answer') or '').strip():
        raise ValueError(f'Mẫu train {sample_id!r} thiếu question/answer hợp lệ')
TRAIN10_IDS = [str(sample_id) for sample_id, _ in TRAIN10_ITEMS]
TRAIN10_DATA = {str(sample_id): {'question': item['question'], 'answer': item['answer']} for sample_id, item in TRAIN10_ITEMS}
TRAIN10_PATH = WORK_ROOT / f'train_first10_{COMMIT_SHA[:12]}.json'
write_json(TRAIN10_PATH, TRAIN10_DATA)

import pandas as pd
display(pd.DataFrame([
    {
        'order': number,
        'id': sample_id,
        'question': TRAIN10_DATA[sample_id]['question'],
        'reference_words': len(TRAIN10_DATA[sample_id]['answer'].split()),
    }
    for number, sample_id in enumerate(TRAIN10_IDS, start=1)
]))
print('Train10 IDs:', TRAIN10_IDS)
print('Input:', TRAIN10_PATH)

In [ ]:
# 4) Leave-one-out validation: test đủ bốn mode, không cho KNN dùng lại chính ID.
REPORT_PATH = WORK_ROOT / f'pipeline_smoke_train10_{COMMIT_SHA[:12]}.json'
RUN_LOG_PATH = WORK_ROOT / f'pipeline_smoke_train10_{COMMIT_SHA[:12]}.log'
PIPELINE_ARGS = [
    '--db', str(LOCAL_DB_PATH),
    '--top-k', '12',
    '--bm25-top-k', str(BM25_TOP_K),
    '--dense-top-k', str(DENSE_TOP_K),
    '--rrf-k', '60',
    '--rrf-top-k', str(RRF_TOP_K),
    '--reranker-candidate-k', str(RERANKER_CANDIDATE_K),
    '--rerank-top-k', str(RERANK_TOP_K),
    '--context-top-k', str(RERANK_TOP_K),
    '--dense-query-max-length', '256',
    '--reranker-max-length', str(RERANKER_MAX_LENGTH),
    '--knn-threshold', '0.72',
    '--guarded-knn-threshold', '0.90',
    '--dense-index', str(DENSE_INDEX_PATH),
    '--embedding-model', EMBEDDING_MODEL_ID,
    '--embedding-revision', MODEL_INFO['embedding']['revision'],
    '--reranker-model', str(MODEL_INFO['reranker']['path']),
    '--generator-model', str(MODEL_INFO['generator']['path']),
    '--device', 'auto',
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--max-input-tokens', str(MAX_INPUT_TOKENS),
    '--token-limit-retry-tokens', str(TOKEN_LIMIT_RETRY_TOKENS),
    '--max-long-answer-words', str(MAX_LONG_ANSWER_WORDS),
    '--generation-seed', str(SEED),
]
if FORCE_RERUN or not REPORT_PATH.is_file():
    command = [
        sys.executable, '-m', 'legalqa_baseline', 'validate',
        '--train', str(TRAIN10_PATH),
        '--output', str(REPORT_PATH),
        '--modes', ','.join(MODES),
        '--limit', str(TRAIN_LIMIT),
        '--seed', str(SEED),
        *PIPELINE_ARGS,
    ]
    started = time.perf_counter()
    run_stream(command, cwd=REPO_DIR, log_path=RUN_LOG_PATH)
    print(f'Pipeline smoke elapsed: {time.perf_counter() - started:.1f}s')
else:
    print('Reuse:', REPORT_PATH)

In [ ]:
# 5) Tổng hợp lỗi kỹ thuật và các câu cần duyệt chất lượng.
from legalqa_baseline.text import (
    is_heading_only_answer,
    is_refusal_answer,
    output_artifact_flags,
    possibly_cut,
)

report = read_json(REPORT_PATH)
reported_ids = {str(value) for value in report.get('config', {}).get('sample_ids', [])}
input_ids_ok = reported_ids == set(TRAIN10_IDS)
dense_active = isinstance(report.get('config', {}).get('dense_index'), dict)
summary_rows: list[dict[str, Any]] = []
technical_errors: list[dict[str, Any]] = []
quality_review: list[dict[str, Any]] = []
mode_summaries: dict[str, Any] = {}

for mode in MODES:
    result = report.get('results', {}).get(mode, {})
    items = result.get('items', []) if isinstance(result, dict) else []
    mode_summaries[mode] = {
        'samples': len(items),
        'meteor_exact_approx': round(float(result.get('meteor_exact_approx', 0.0)), 4),
        'rougeL': round(float(result.get('rougeL', 0.0)), 4),
        'answer_token_f1': round(float(result.get('answer_token_f1', 0.0)), 4),
        'routes': result.get('routes', {}),
        'routing_quality': result.get('routing_quality', {}),
    }
    if len(items) != TRAIN_LIMIT:
        technical_errors.append({'mode': mode, 'id': None, 'errors': [f'item_count={len(items)}']})
    for item in items:
        sample_id = str(item.get('id'))
        prediction = str(item.get('prediction') or '').strip()
        audit = item.get('audit', {}) if isinstance(item.get('audit'), dict) else {}
        metrics = item.get('metrics', {}) if isinstance(item.get('metrics'), dict) else {}
        flags = sorted(output_artifact_flags(prediction))
        refusal = bool(is_refusal_answer(prediction) or audit.get('says_no_information'))
        token_limit = bool(audit.get('hit_token_limit'))
        cut = bool(possibly_cut(prediction) or audit.get('possibly_cut'))
        heading = bool(str(item.get('route', '')).startswith('extractive') and is_heading_only_answer(prediction))
        errors = []
        if not prediction:
            errors.append('empty')
        if refusal:
            errors.append('refusal')
        if token_limit:
            errors.append('token_limit')
        if cut:
            errors.append('possibly_cut')
        if heading:
            errors.append('heading_only')
        errors.extend(f'artifact:{flag}' for flag in flags)
        meteor = float(metrics.get('meteor_exact_approx', 0.0))
        rouge = float(metrics.get('rougeL', 0.0))
        low_similarity = meteor < 0.15 and rouge < 0.15
        row = {
            'mode': mode,
            'order': TRAIN10_IDS.index(sample_id) + 1 if sample_id in TRAIN10_IDS else None,
            'id': sample_id,
            'route': item.get('route'),
            'errors': ','.join(errors),
            'low_similarity': low_similarity,
            'meteor': round(meteor, 4),
            'rougeL': round(rouge, 4),
            'prediction_words': item.get('length', {}).get('prediction_words'),
            'reference_words': item.get('length', {}).get('reference_words'),
            'seconds': round(float(audit.get('stage_seconds', {}).get('total', 0.0)), 2),
            'question': item.get('question'),
            'prediction_preview': prediction[:220],
            'reference_preview': str(item.get('reference') or '')[:220],
        }
        summary_rows.append(row)
        if errors:
            technical_errors.append({'mode': mode, 'id': sample_id, 'errors': errors})
        if low_similarity:
            quality_review.append({'mode': mode, 'id': sample_id, 'meteor': meteor, 'rougeL': rouge})

required_retrieval_stages = {'bm25', 'dense', 'rrf', 'reranker_pool', 'reranker_top'}
hybrid_rag_items = report.get('results', {}).get('hybrid_rag', {}).get('items', [])
retrieval_stage_ids = []
incomplete_retrieval_ids = []
for item in hybrid_rag_items:
    if str(item.get('route', '')).startswith('knn_'):
        continue
    trace = item.get('retrieval', {}).get('trace', {})
    completed = {
        stage
        for stage in required_retrieval_stages
        if isinstance(trace.get(stage), dict)
        and trace[stage].get('status') not in {'error', 'unavailable'}
        and isinstance(trace[stage].get('candidates'), list)
        and trace[stage]['candidates']
    }
    if completed == required_retrieval_stages:
        retrieval_stage_ids.append(str(item.get('id')))
    else:
        incomplete_retrieval_ids.append({
            'id': str(item.get('id')),
            'missing_stages': sorted(required_retrieval_stages - completed),
        })

hard_gates = {
    'first_10_ids_exact': input_ids_ok,
    'dense_index_active': dense_active,
    'all_modes_have_10_items': all(mode_summaries.get(mode, {}).get('samples') == TRAIN_LIMIT for mode in MODES),
    'hybrid_rag_full_retrieval_seen': bool(retrieval_stage_ids),
    'hybrid_rag_retrieval_stages_complete': not incomplete_retrieval_ids,
    'no_technical_output_errors': not technical_errors,
}
status = 'PIPELINE_SMOKE_PASS' if all(hard_gates.values()) else 'PIPELINE_SMOKE_FAIL'
compact_report = {
    'status': status,
    'commit_sha': COMMIT_SHA,
    'train10_ids': TRAIN10_IDS,
    'hard_gates': hard_gates,
    'mode_summaries': mode_summaries,
    'retrieval_stage_ids': retrieval_stage_ids,
    'incomplete_retrieval_ids': incomplete_retrieval_ids,
    'technical_errors': technical_errors,
    'quality_review': quality_review,
    'note': 'Low similarity là cờ duyệt tay trên 10 mẫu nhỏ, không tự động là lỗi pipeline.',
}
SUMMARY_PATH = WORK_ROOT / f'pipeline_smoke_train10_summary_{COMMIT_SHA[:12]}.json'
write_json(SUMMARY_PATH, compact_report)

display(pd.DataFrame(summary_rows).sort_values(['mode', 'order']))
print(json.dumps(compact_report, ensure_ascii=False, indent=2))
print('Full report:', REPORT_PATH)
print('Summary:', SUMMARY_PATH)
print('Log:', RUN_LOG_PATH)
print('KẾT LUẬN:', status)
if status == 'PIPELINE_SMOKE_PASS':
    print('Pipeline không có lỗi kỹ thuật trên train10. Duyệt các quality_review trước khi chạy smoke30/validation lớn hơn.')
else:
    print('Dừng tại train10 và sửa technical_errors; chưa chạy smoke30 hoặc full 1.000.')

## File đầu ra

- `train_first10_<sha>.json`: đúng 10 mẫu train đầu được dùng.
- `pipeline_smoke_train10_<sha>.json`: report đầy đủ, gồm prediction/reference/metric/audit từng mode.
- `pipeline_smoke_train10_summary_<sha>.json`: gate, lỗi kỹ thuật và danh sách cần duyệt chất lượng.
- `pipeline_smoke_train10_<sha>.log`: log chạy CLI.

Chỉ tăng lên smoke 30 sau khi `PIPELINE_SMOKE_PASS`; không dùng điểm trên 10 câu đầu làm ước lượng chất lượng toàn bộ tập.